In [66]:
# --- IMPORT STATEMENTS ---
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
import healpy as hp

In [67]:
# --- GLOBAL VARIABLES ---

LINUX_DIRECTORY = "/home/aimee/mphys" # Aimee
MAPS_DIRECTORY = f'{LINUX_DIRECTORY}/data/maps'
MAP_NAMES = ['Effelsberg_2.7272GHz_raw.fits', 'Nobeyama_10.0GHz_raw.fits', 'Parkes_5.0GHz_raw.fits']

SAVE_FIGS = False

In [68]:
def get_map(path):
    
    print(f"Getting data from {path}")
    fits_file = fits.open(path)

    data = fits_file[0].data
    header = fits_file[0].header
    wcs = WCS(header)
        
    return data, header, wcs

In [69]:
def plot_fits(fits_data, projection, x_coords=[], y_coords=[], title='Title', save_figs=SAVE_FIGS): # modified from jg_notebooks/gp_plotting.ipynb
    
    fits_data = np.where(fits_data == hp.UNSEEN, np.nan, fits_data) # convert hp.UNSEEN to np.nan for plotting
    
    vmin = np.nanpercentile(fits_data, 0.)
    vmax = np.nanpercentile(fits_data, 99.5)

    fig, ax = plt.subplots(1, 1, figsize=(13,2), dpi=300,
                            subplot_kw={'projection':projection})

    im = ax.imshow(fits_data, origin='lower', cmap='viridis', vmin=vmin , vmax=vmax)
    
    if x_coords is not None and y_coords is not None: # optional: plot provided coordinates over image
        ax.plot(x_coords, y_coords, 'rx', markersize=3)
    
    ax.set_title(title)
    ax.set_xlabel(r"$l$ [degrees]")
    ax.set_ylabel(r"$b$ [degrees]")

    cbar = fig.colorbar(im, ax=ax, orientation='vertical')
    cbar.set_label(r"$T_b$ [K]")

    fig.subplots_adjust(left=0.07, right=0.88, top=0.95, bottom=0.05, hspace=0.4)
    plt.tight_layout()

    if save_figs:
        plt.savefig(f"{FIGURE_SAVEDIR}/{title.replace(' ', '_')}.png", dpi=300)
    
    plt.show()

In [70]:
def reproject_data(data, wcs, shape_out, pixel_scale_deg): # use with caution
   
    wcs = wcs.sub([0, 1]) # Slice the WCS to 2D to ignore the frequency axis

    # Get the center of the original image (mosaic_wcs)
    center_pix = np.array(shape_out) / 2
    center_lon, center_lat = wcs.pixel_to_world_values(center_pix[1], center_pix[0]) 

    # Determine angular size of the original image (in degrees)
    orig_scales = proj_plane_pixel_scales(wcs)  # degrees/pixel in x and y
    size_deg_x = orig_scales[0] * shape_out[1]
    size_deg_y = orig_scales[1] * shape_out[0]

    # Compute new shape in pixels for the same area with 1 arcmin/pixel
    new_shape_x = int(np.ceil(size_deg_x / pixel_scale_deg))
    new_shape_y = int(np.ceil(size_deg_y / pixel_scale_deg))
    new_shape = (new_shape_y, new_shape_x)

    # Create a new WCS
    new_wcs = WCS(naxis=3)
    new_wcs.wcs.crval = [center_lon, center_lat]              # center of the image in degrees
    new_wcs.wcs.crpix = [new_shape[1] / 2, new_shape[0] / 2]  # center pixel for new shape
    new_wcs.wcs.ctype = list(wcs.wcs.ctype)           # keep original projection
    new_wcs.wcs.cdelt = np.array([-pixel_scale_deg, pixel_scale_deg])  # pixel scale in degrees

    # Reproject the mosaic to the new WCS
    data_reprojected, footprint = reproject_interp(
        (data, wcs),
        new_wcs,
        shape_out=new_shape
    )

    # Convert the WCS to a FITS header
    new_header = new_wcs.to_header()

    return data_reprojected, new_wcs, new_header

In [71]:
def rebin_array(data, factor):
    n_rows, n_cols = data.shape
    # print(f"Original shape: {data.shape}")

    # Crop rows and columns if not divisible by factor
    if n_rows % factor != 0:
        new_n_rows = (n_rows // factor) * factor
        data = data[:new_n_rows, :]
        # print(f"Adjusted rows to: {new_n_rows}")
    if n_cols % factor != 0:
        new_n_cols = (n_cols // factor) * factor
        data = data[:, :new_n_cols]
        # print(f"Adjusted columns to: {new_n_cols}")

    # Reshape and average in blocks of factor x factor
    reshaped_data = data.reshape(data.shape[0] // factor, factor, data.shape[1] // factor, factor)
    rebinned_data = reshaped_data.mean(axis=(1, 3))  # If any pixel in block is nan, whole block is nan (good)
    # print(f"Re-shaped data to: {rebinned_data.shape}")
    
    return rebinned_data


def rebin_map(data, wcs, factor):
    
    l_min = L_BOUNDS[0]
    l_max = L_BOUNDS[1]
    b_min = B_BOUNDS[0]
    b_max = B_BOUNDS[1]
    step = wcs.wcs.cdelt[1] # pixel size in degrees

    data = rebin_array(data, factor)

    ny, nx = data.shape
    cy, cx = ny // 2, nx // 2
    # print(cy, cx)   

    # Indices for cutout
    new_wcs = wcs.copy()
    new_wcs.wcs.crpix[0] = cx
    new_wcs.wcs.crpix[1] = cy
    new_wcs.wcs.cdelt[0] = -step * factor
    new_wcs.wcs.cdelt[1] = step * factor
    new_wcs.wcs.crval[0] = (l_min + l_max) / 2
    new_wcs.wcs.crval[1] = (b_min + b_max) / 2

    new_header = new_wcs.to_header()

    return data, new_wcs, new_header

In [ ]:
# --- MAIN CODE ---

map_name = MAP_NAMES[0]
data, header, wcs = get_map(f'{MAPS_DIRECTORY}/{map_name}')

# # Oversample reproject (fixes 3D wcs)
pixel_scale_deg = 1 / 60  #  get from header instead!
new_pixel_scale_deg = pixel_scale_deg / 2 

shape_out = np.array(data.shape) * 2 # double resolution

reprojected_data, reprojected_wcs, reprojected_header = reproject_data(
    data, wcs, shape_out=shape_out, pixel_scale_deg=new_pixel_scale_deg
)
print(f"Reprojected data shape: {reprojected_data.shape}")
print(f"Reprojected WCS dimensions: {reprojected_wcs.naxis}")

plot_fits(reprojected_data, reprojected_wcs, title='Reprojected (Oversampled) Data')

# # Mean back to original resolution
rebinned_data, rebinned_wcs, rebinned_header = rebin_map(data, wcs, header)
plot_fits(rebinned_data, rebinned_wcs, title='Rebinned Data (Back to Original Resolution)')


# Save with to MAPS_DIRECTORY with suffix _2Dwcs
# hdu = fits.PrimaryHDU(data=rebinned_data, header=rebinned_header)
# hdu.header['BUNIT'] = 'K'
# hdu.writeto(f'{MAPS_DIRECTORY}/{map_name.replace(".fits", "")}_2Dwcs.fits', overwrite=True)


Getting data from /home/aimee/mphys/data/maps/Effelsberg_2.7272GHz_raw.fits


ValueError: 'crval' array is the wrong shape, must be 3